In [74]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import datetime
from sklearn.preprocessing import LabelEncoder,OneHotEncoder

# 불필요한 경고문 생략(선택)
import warnings
warnings.filterwarnings('ignore')

# 모든 컬럼 출력설정(선택)
pd.set_option('display.max_columns', None)

#데이터 불러오기 
df = pd.read_csv('model_df_new_cat_음수할인율처리후_이월추가.csv')

In [75]:
#한글 
import platform

def set_matplotlib_font():
    system = platform.system()

    if system == "Windows":
        plt.rc('font', family='Malgun Gothic')
    elif system == "Darwin":  # macOS
        plt.rc('font', family='AppleGothic')
    elif system == "Linux":
        plt.rc('font', family='NanumGothic')
    else:
        print("Unknown system. Please set font manually.")

    plt.rcParams['axes.unicode_minus'] = False

# 폰트 설정 함수 호출
set_matplotlib_font()

In [76]:
#주차 컬럼 날짜타입 변환 (범주->날짜형)
df['주차'] = pd.to_datetime(df['주차'])

In [77]:
# 데이터 확인 
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5583 entries, 0 to 5582
Data columns (total 26 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   기획년도         5583 non-null   int64         
 1   주차           5583 non-null   datetime64[ns]
 2   카테고리_통합      5583 non-null   object        
 3   시즌이월         5583 non-null   object        
 4   시즌           5583 non-null   object        
 5   총입고수량        5583 non-null   int64         
 6   판매수량         5583 non-null   int64         
 7   판매액          5583 non-null   int64         
 8   평균택가         5583 non-null   int64         
 9   평균원가         5583 non-null   int64         
 10  주차별_평균_실판매가  5412 non-null   float64       
 11  월별_평균_실판매가   5545 non-null   float64       
 12  시즌별_평균_실판매가  5583 non-null   float64       
 13  총입고원가        5583 non-null   int64         
 14  총입고택가        5583 non-null   int64         
 15  매출원가계        5583 non-null   int64         
 16  판매택가계 

기획년도             0
주차               0
카테고리_통합          0
시즌이월             0
시즌               0
총입고수량            0
판매수량             0
판매액              0
평균택가             0
평균원가             0
주차별_평균_실판매가    171
월별_평균_실판매가      38
시즌별_평균_실판매가      0
총입고원가            0
총입고택가            0
매출원가계            0
판매택가계            0
실판매가             0
할인율              0
누적판매수량           0
누적판매액            0
누적매출원가           0
누적판매택가           0
누적판매율            0
ROI              0
평균기온(도)          0
dtype: int64

In [78]:
df.head(3)

,기획년도,주차,카테고리_통합,시즌이월,시즌,총입고수량,판매수량,판매액,평균택가,평균원가,주차별_평균_실판매가,월별_평균_실판매가,시즌별_평균_실판매가,총입고원가,총입고택가,매출원가계,판매택가계,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI,평균기온(도)
0,2021,2021-07-11,가을_가성비 수트셋업,01_시즌,가을,13163,496,84636400,184000,28105,182554.0,138940.0,139804.0,369946115,2421992000,13940080,91264000,170638,7,496,84636400,13940080,91264000,3.77,0.17,28.5
1,2021,2021-07-18,가을_가성비 수트셋업,01_시즌,가을,13163,449,56822086,184000,28105,94914.0,138940.0,139804.0,369946115,2421992000,12619145,82616000,126553,31,945,141458486,26559225,173880000,7.18,0.28,29.8
2,2021,2021-07-25,가을_가성비 수트셋업,01_시즌,가을,15747,753,99929738,184000,28218,135804.0,138940.0,139804.0,444348846,2897448000,21248154,138552000,132709,28,1698,241388224,47807379,312432000,10.78,0.39,30.6


# 주간 입고 데이터 추가 

In [79]:
# 주간 입고 데이터 추가 
df= df.sort_values(by='주차')
df['입고수량'] = df.groupby(['기획년도','카테고리_통합'])['총입고수량'].diff().fillna(df['총입고수량'])
df['입고원가계'] = df.groupby(['기획년도','카테고리_통합'])['총입고원가'].diff().fillna(df['총입고원가'])
df['입고택가계'] = df.groupby(['기획년도','카테고리_통합'])['총입고택가'].diff().fillna(df['총입고택가'])
df.head(3)

,기획년도,주차,카테고리_통합,시즌이월,시즌,총입고수량,판매수량,판매액,평균택가,평균원가,주차별_평균_실판매가,월별_평균_실판매가,시즌별_평균_실판매가,총입고원가,총입고택가,매출원가계,판매택가계,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI,평균기온(도),입고수량,입고원가계,입고택가계
1299,2021,2021-01-03,봄_가디건,01_시즌,봄,8976,127,14957130,159900,14254,108352.0,152746.0,79061.0,127943904,1435262400,1810258,20307300,117773,26,127,14957130,1810258,20307300,1.41,0.09,-9.1,8976.0,127943904.0,1.435262e+09
1507,2021,2021-01-03,봄_가성비 수트셋업,01_시즌,봄,24728,498,56952161,165667,32868,135805.0,118744.0,97673.0,812759904,4096613576,16368264,82502166,114362,31,498,56952161,16368264,82502166,2.01,0.04,-9.1,24728.0,812759904.0,4.096614e+09
1716,2021,2021-01-03,봄_고가 수트셋업,01_시즌,봄,9635,237,32607290,265666,44136,162746.0,146322.0,116112.0,425250360,2559691910,10460232,62962842,137584,48,237,32607290,10460232,62962842,2.46,0.05,-9.1,9635.0,425250360.0,2.559692e+09


# 목표 판매율 도출 

In [85]:
# ============================================================
# 1. 연마감(기획년도, 시즌) 총 판매수량 및 총 입고수량 집계
# ============================================================
all_goal_df = df.groupby(['기획년도', '시즌']).agg(
    총판매수량=("판매수량", "sum"),
    총입고수량=("입고수량", "sum")
).reset_index()

# 연 마감 판매율 및 잔여 재고량 계산 (백분율, 정수형 재고량)
all_goal_df['연마감 판매율'] = (all_goal_df['총판매수량'] / all_goal_df['총입고수량'] * 100).round(1)
all_goal_df['연마감 재고량'] = (all_goal_df['총입고수량'] - all_goal_df['총판매수량']).astype('int64')



# *** 2023년도 연마감 데이터 추출 및 컬럼명 추가
df2_2023 = all_goal_df[all_goal_df['기획년도'] == 2023][['시즌', '연마감 재고량']]
df2_2023.rename(columns={'연마감 재고량': '2023_연마감재고량'}, inplace=True)


# ============================================================
# 2. 시즌마감 데이터 계산
#    - '시즌이월' 값이 '01_시즌'인 데이터만 사용
# 기획년도와 시즌별로 총 판매수량, 총 입고수량 집계
# ============================================================
season_df = df[df['시즌이월'] == '01_시즌']

season_goal_df = season_df.groupby(['기획년도', '시즌']).agg(
    총판매수량=("판매수량", "sum"),
    총입고수량=("입고수량", "sum")
).reset_index()

# 시즌마감 판매율 및 잔여 재고량 계산
season_goal_df['시즌마감 판매율'] = (season_goal_df['총판매수량'] / season_goal_df['총입고수량'] * 100).round(1)
season_goal_df['시즌마감 재고량'] = (season_goal_df['총입고수량'] - season_goal_df['총판매수량']).astype('int64')

# 시즌별 판매율 변화량 계산 (전년도 대비 변화)
season_goal_df["판매율 변화량"] = season_goal_df.groupby("시즌")["시즌마감 판매율"].diff()
print(season_goal_df)


# ============================================================
# 3. 2023년도 시즌마감 데이터 추출 및 컬럼명 변경
# ============================================================
df_2023 = season_goal_df[season_goal_df['기획년도'] == 2023].copy()
df_2023.rename(columns={
    '시즌마감 판매율': '2023_시즌마감판매율',
    '시즌마감 재고량': '2023_시즌마감재고량',
    '총판매수량': '2023_총판매량'
}, inplace=True)

#컬럼 필터링
df_2023.drop(columns=['기획년도', '판매율 변화량', '총입고수량'], inplace=True)

# *** 2024년도 시즌마감 판매율 추가
df_2024 = season_goal_df[season_goal_df['기획년도'] == 2024].copy()
df_2024.rename(columns={
    '시즌마감 판매율': '2024_시즌마감판매율'
}, inplace=True)

#컬럼 필터링
df_2024.drop(columns=['기획년도', '판매율 변화량', '총입고수량', '총판매수량','시즌마감 재고량','판매율 변화량'], inplace=True)

# ============================================================
# 4. 시즌 평균치 및 연 평균치 계산
# ============================================================
# 시즌별 평균치 계산 (판매율, 재고량, 총 판매량 및 판매율 변화량의 변화 평균)
season_avg = season_goal_df.groupby("시즌").agg(
    시즌마감_평균판매율=("시즌마감 판매율", "mean"),
    시즌마감_판매율변화량=("판매율 변화량", lambda x: x.diff().mean()),
    시즌마감_평균재고량=("시즌마감 재고량", "mean"),
    시즌마감_총판매량=("총판매수량", "mean")
).reset_index()

# 연도별 평균치 계산 (연마감 기준)
all_avg = all_goal_df.groupby("시즌").agg(
    연_평균판매율=("연마감 판매율", "mean"),
    연_평균재고량=("연마감 재고량", "mean"),
    연_총판매량=("총판매수량", "mean")
).reset_index()



# ============================================================
# 5. 데이터 병합
#    - 시즌별 평균치, 2023 시즌마감 데이터, 연평균 데이터, 2023 연마감 재고량 병합
# ============================================================
# 시즌평균과 2023시즌마감 /2024 판매율 데이터 병합
season_add = season_avg.merge(df_2023, on='시즌', how='left')
merge_df = season_add.merge(df_2024, on='시즌', how='left')

# 연평균 데이터 병합
merge_df2 = merge_df.merge(all_avg, on='시즌', how='left')
# 2023 연마감 재고량 병합
merge_df3 = merge_df2.merge(df2_2023, on='시즌', how='left')

# 원하는 컬럼 순서로 재정렬
final_merge_df = merge_df3[['시즌',
                            '시즌마감_판매율변화량',
                            '2023_시즌마감판매율',
                            '시즌마감_평균판매율',
                            '연_평균판매율',
                            '2023_시즌마감재고량',
                            '2023_연마감재고량',
                            '시즌마감_평균재고량',
                            '연_평균재고량',
                            '2024_시즌마감판매율']]


# 시즌-마감 판매율 차이 계산 (연평균 판매율 - 시즌마감 평균 판매율)
final_merge_df['시즌-마감 판매율 차이(p)'] = final_merge_df['연_평균판매율'] - final_merge_df['시즌마감_평균판매율']

# ============================================================
# 6. 가중치 계산
#    - 2023 시즌마감 재고량 총합, 재고 비중, 조건에 따른 가중치 부여
#    - 겨울 /사계절 제외 (재고 수준 감안할 필요없으므로) 
# ============================================================
# 2023년 시즌마감 재고량 총합 계산
total_season_end_stock = final_merge_df["2023_시즌마감재고량"].sum()

# 각 시즌이 차지하는 2023 시즌마감 재고 비중(%) 계산
final_merge_df["2023_시즌마감재고 비중(%)"] = (final_merge_df["2023_시즌마감재고량"] / total_season_end_stock) * 100

# 겨울과 사계절 데이터 제외
filtered_df = final_merge_df[~final_merge_df["시즌"].isin(["겨울", "사계절"])].copy()

# 조건 1: 2023 시즌마감 재고량이 시즌마감 평균재고량보다 큰 경우 가중치 +2
filtered_df["가중치"] = 0  # 기본 가중치 0으로 초기화
filtered_df.loc[filtered_df["2023_시즌마감재고량"] > filtered_df["시즌마감_평균재고량"], "가중치"] += 2

# 조건 2: 2023 시즌마감별 재고 비중이 큰 순서대로 순위별 가중치 부여 (높은 순위부터 1씩 부여)
filtered_df = filtered_df.sort_values(by="2023_시즌마감재고 비중(%)", ascending=False)
filtered_df["순위별 가중치"] = range(len(filtered_df), 0, -1)

# 최종 가중치는 기본 가중치와 순위별 가중치 합산
filtered_df["최종 가중치"] = filtered_df["가중치"] + filtered_df["순위별 가중치"]

# 최종 가중치 데이터를 기존 데이터와 병합 (겨울, 사계절은 NaN -> 0으로 채움)
final_merge_df2 = final_merge_df.merge(filtered_df[["시즌", "최종 가중치"]], on="시즌", how="left")
final_merge_df2["최종 가중치"] = final_merge_df2["최종 가중치"].fillna(0)

#의지치 5 반영 
final_merge_df2["의지치"] = 5
# ============================================================
# 7. 목표 판매율 도출 및 gap 계산
#    - 계산식: 2023_시즌마감판매율 + 시즌마감 평균 판매율 변화량 +  최종 가중치
# ============================================================
final_merge_df2['24목표 판매율'] = (final_merge_df2['2023_시즌마감판매율'] +
                                    final_merge_df2['시즌마감_판매율변화량'] +
                                    final_merge_df2['의지치'] +
                                    final_merge_df2["최종 가중치"])

# 목표 판매율과 2023 시즌마감 판매율 차이 계산
final_merge_df2['23시즌- 24목표 gap(p)'] = final_merge_df2['24목표 판매율'] - final_merge_df2['2023_시즌마감판매율']
final_merge_df2


    기획년도   시즌   총판매수량      총입고수량  시즌마감 판매율  시즌마감 재고량  판매율 변화량
0   2021   가을   70385   125629.0      56.0     55244      NaN
1   2021   겨울  307420   640554.0      48.0    333134      NaN
2   2021    봄  141824   221776.0      63.9     79952      NaN
3   2021  사계절  214943   286904.0      74.9     71961      NaN
4   2021   여름  917264  1561614.0      58.7    644350      NaN
5   2022   가을   39587    74885.0      52.9     35298     -3.1
6   2022   겨울  223517   539232.0      41.5    315715     -6.5
7   2022    봄  108809   243334.0      44.7    134525    -19.2
8   2022  사계절  255064   386739.0      66.0    131675     -8.9
9   2022   여름  709481  1268385.0      55.9    558904     -2.8
10  2023   가을   41468    71172.0      58.3     29704      5.4
11  2023   겨울  180965   419411.0      43.1    238446      1.6
12  2023    봄   53632   115419.0      46.5     61787      1.8
13  2023  사계절  204424   332769.0      61.4    128345     -4.6
14  2023   여름  497073   708568.0      70.2    211495     14.3
15  2024

,시즌,시즌마감_판매율변화량,2023_시즌마감판매율,시즌마감_평균판매율,연_평균판매율,2023_시즌마감재고량,2023_연마감재고량,시즌마감_평균재고량,연_평균재고량,2024_시즌마감판매율,시즌-마감 판매율 차이(p),2023_시즌마감재고 비중(%),최종 가중치,의지치,24목표 판매율,23시즌- 24목표 gap(p)
0,가을,1.5,58.3,56.35,62.250,29704,23508,39239.00,34156.00,58.2,5.900,4.434909,1.0,5,65.8,7.5
1,겨울,6.8,43.1,45.70,45.700,238446,238446,268647.50,268647.50,50.2,0.000,35.600804,0.0,5,54.9,11.8
2,봄,23.1,46.5,57.15,75.600,61787,33641,74858.25,43570.00,73.5,18.450,9.225011,2.0,5,76.6,30.1
3,사계절,9.9,61.4,68.65,68.650,128345,128345,102441.75,102441.75,72.3,0.000,19.162348,0.0,5,76.3,14.9
4,여름,4.7,70.2,65.40,67.575,211495,183718,385528.50,364658.75,76.8,2.175,31.576928,3.0,5,82.9,12.7


In [86]:
#24년도 gap 추가
final_merge_df2['24 실제-목표 gap(p)'] = final_merge_df2['24목표 판매율'] - final_merge_df2['2024_시즌마감판매율']

#최종 필터링
final_merge_df2[['시즌','2023_시즌마감판매율','시즌마감_판매율변화량','최종 가중치','의지치','24목표 판매율','23시즌- 24목표 gap(p)','2024_시즌마감판매율','24 실제-목표 gap(p)']]

,시즌,2023_시즌마감판매율,시즌마감_판매율변화량,최종 가중치,의지치,24목표 판매율,23시즌- 24목표 gap(p),2024_시즌마감판매율,24 실제-목표 gap(p)
0,가을,58.3,1.5,1.0,5,65.8,7.5,58.2,7.6
1,겨울,43.1,6.8,0.0,5,54.9,11.8,50.2,4.7
2,봄,46.5,23.1,2.0,5,76.6,30.1,73.5,3.1
3,사계절,61.4,9.9,0.0,5,76.3,14.9,72.3,4.0
4,여름,70.2,4.7,3.0,5,82.9,12.7,76.8,6.1
